# Sprint 2 — Perfil do dataset bruto Favorita

Chamador fino: carrega o parquet bruto (`motor.io.raw.scan_raw`) e chama as funções de `motor.profiling`. Nenhuma agregação nova é escrita aqui — se uma célula precisasse de uma, ela pertenceria a `profiling.py`.

Nenhuma transformação acontece antes deste ponto: os nomes de coluna são os do arquivo bruto (`store_nbr`, `item_nbr`, `unit_sales`, `onpromotion`), e nenhuma linha foi filtrada, reindexada ou imputada.

In [1]:
from pathlib import Path

import polars as pl

from motor.config import load_params
from motor.io.raw import scan_raw
from motor.profiling import (
    count_candidates,
    profile_calendar,
    profile_fractional_by_item,
    profile_fractional_global,
    profile_item_volume,
    profile_negatives_by_item,
    profile_negatives_global,
    profile_pair_coverage,
    profile_promo_availability,
    profile_store_lifespan,
)

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)

REPO_ROOT = Path.cwd().parent
params = load_params(REPO_ROOT / "config" / "params.yaml")
train = scan_raw("train", REPO_ROOT / params.data.raw_parquet_dir)
items = scan_raw("items", REPO_ROOT / params.data.raw_parquet_dir)

## Calendário e cobertura global

In [2]:
profile_calendar(train)

date_min,date_max,n_stores,n_items,n_rows,n_pairs
date,date,u32,u32,u32,i32
2013-01-01,2017-08-15,54,4036,125497040,174685


In [3]:
profile_store_lifespan(train)

store_nbr,first_sale,last_sale,n_days_with_movement
i64,date,date,u32
1,2013-01-02,2017-08-15,1678
2,2013-01-02,2017-08-15,1679
3,2013-01-02,2017-08-15,1679
4,2013-01-02,2017-08-15,1679
5,2013-01-02,2017-08-15,1679
6,2013-01-02,2017-08-15,1679
7,2013-01-02,2017-08-15,1679
8,2013-01-02,2017-08-15,1679
9,2013-01-02,2017-08-15,1679


## Vendas negativas e fracionárias

Nenhuma das duas é tratada nesta sprint (armadilha: "venda negativa" e "venda fracionária" viram, respectivamente, devolução e `unit_of_sale = kg` só na Sprint 3). Aqui é só medição.

In [4]:
profile_negatives_global(train)

n_rows,n_negative_rows,negative_ratio
u32,u32,f64
125497040,7795,0.000062


In [5]:
profile_negatives_by_item(train).sort("negative_ratio", descending=True).head(15)

item_nbr,n_rows,n_negative_rows,negative_ratio
i64,u32,u32,f64
2121610,13,1,0.076923
2006144,3547,66,0.018607
1316461,871,8,0.009185
1956062,1965,16,0.008142
1956622,2215,16,0.007223
2011898,1386,8,0.005772
1412379,3423,16,0.004674
2049026,457,2,0.004376
172995,15986,69,0.004316


In [6]:
profile_fractional_global(train)

n_rows,n_fractional_rows,fractional_ratio
u32,u32,f64
125497040,8191095,0.065269


In [7]:
profile_fractional_by_item(train).sort("fractional_ratio", descending=True).head(15)

item_nbr,n_rows,n_fractional_rows,fractional_ratio
i64,u32,u32,f64
205209,11028,11021,0.999365
2057231,7703,7698,0.999351
1109211,2928,2926,0.999317
844019,48170,48128,0.999128
699745,29323,29297,0.999113
1288843,19009,18992,0.999106
583925,38706,38671,0.999096
507354,37157,37123,0.999085
584076,38146,38111,0.999082


## Disponibilidade de `onpromotion`

Nulo por mês — é o que decide a partir de quando o campo é utilizável.

In [8]:
profile_promo_availability(train)

month,n_rows,n_null,null_ratio
date,u32,u32,f64
2013-01-01,1211964,1211964,1.0
2013-02-01,1160386,1160386,1.0
2013-03-01,1321927,1321927,1.0
2013-04-01,1284375,1284375,1.0
2013-05-01,1360917,1360917,1.0
2013-06-01,1353864,1353864,1.0
2013-07-01,1385579,1385579,1.0
2013-08-01,1421744,1421744,1.0
2013-09-01,1378438,1378438,1.0


## Volume por item (em UNIDADES)

Não existe preço nem custo no Favorita — "participação" aqui é sempre participação em unidades vendidas, nunca em R$.

In [9]:
profile_item_volume(train).head(20)

item_nbr,total_units,median_daily_units,p90_daily_units,max_daily_units,n_stores,cum_unit_share
i64,f64,f64,f64,f64,u32,f64
1503844,6.2642e6,163.163,516.402,1512.857,43,0.005835
1047679,5.512875e6,25.0,183.0,6131.0,54,0.01097
1473474,4.9903e6,61.378,271.452,1001.438,53,0.015618
364606,4.416426e6,43.0,101.0,4635.0,54,0.019731
819932,4.410943e6,37.0,136.0,3040.0,54,0.02384
1463992,4.408545e6,50.0,227.0,1122.0,54,0.027946
807493,4.321424e6,33.0,125.0,2578.0,54,0.031971
265559,4.112827e6,37.0,96.0,12228.0,54,0.035802
502331,3.70335e6,38.0,81.0,833.0,54,0.039252


## Cobertura por par loja-item e candidatos ao subconjunto, por janela

Duas janelas candidatas (`config/params.yaml`, seção `profiling.windows`) — o trade-off entre elas (mais histórico vs. terremoto de 16/04/2016) fica visível em número, não é decidido aqui.

In [10]:
coverage_by_window = {}
for window in params.profiling.windows:
    coverage_by_window[window.name] = profile_pair_coverage(
        train, window_start=window.start, window_end=window.end
    )
    n_pairs = coverage_by_window[window.name].height
    print(f"{window.name} ({window.start} -> {window.end}): {n_pairs} pares")

ultimos_12_meses (2016-08-16 -> 2017-08-15): 170803 pares


ultimos_24_meses (2015-08-16 -> 2017-08-15): 172769 pares


In [11]:
window = params.profiling.windows[0]
print(window.name)
coverage_by_window[window.name].describe()

ultimos_12_meses


statistic,store_nbr,item_nbr,first_sale,last_sale,n_days_with_sale,span_days,density_pair_span,density_window,store_covers_full_window
str,f64,f64,str,str,f64,f64,f64,f64,f64
"""count""",170803.0,170803.0,"""170803""","""170803""",170803.0,170803.0,170803.0,170803.0,170803.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0
"""mean""",27.864838,1.2200e6,"""2016-09-17 08:05:01.315550""","""2017-07-29 00:06:12.808440""",218.722042,315.667494,0.679513,0.608995,0.97962
"""std""",15.974464,582719.557212,null,null,111.864329,90.597472,0.269026,0.304659,null
"""min""",1.0,96995.0,"""2016-08-16""","""2016-08-16""",1.0,1.0,0.005731,0.00274,0.0
"""25%""",14.0,795612.0,"""2016-08-16""","""2017-08-13""",121.0,307.0,0.486188,0.356164,null
"""50%""",28.0,1.239908e6,"""2016-08-16""","""2017-08-15""",238.0,364.0,0.75,0.665753,null
"""75%""",42.0,1.695882e6,"""2016-08-22""","""2017-08-15""",323.0,365.0,0.916031,0.890411,null
"""max""",54.0,2.127114e6,"""2017-08-15""","""2017-08-15""",363.0,365.0,1.0,1.0,1.0


In [12]:
window = params.profiling.windows[0]
print(f"{window.name} -- require_full_window=False")
count_candidates(
    coverage_by_window[window.name].lazy(),
    min_density=params.profiling.density_cuts,
    require_full_window=False,
    items=items,
)

ultimos_12_meses -- require_full_window=False


min_density,n_pairs,n_stores,n_items,n_families,top_family
f64,u32,u32,u32,u32,str
0.5,110248,54,3639,32,"""GROCERY I"""
0.6,96123,54,3464,32,"""GROCERY I"""
0.7,79578,53,3267,31,"""GROCERY I"""
0.8,62127,53,3073,26,"""GROCERY I"""
0.9,40280,52,2765,24,"""GROCERY I"""
0.95,25732,52,2419,23,"""GROCERY I"""


In [13]:
window = params.profiling.windows[0]
print(f"{window.name} -- require_full_window=True")
count_candidates(
    coverage_by_window[window.name].lazy(),
    min_density=params.profiling.density_cuts,
    require_full_window=True,
    items=items,
)

ultimos_12_meses -- require_full_window=True


min_density,n_pairs,n_stores,n_items,n_families,top_family
f64,u32,u32,u32,u32,str
0.5,107620,53,3555,32,"""GROCERY I"""
0.6,93722,53,3365,32,"""GROCERY I"""
0.7,77422,52,3155,30,"""GROCERY I"""
0.8,60309,52,2946,24,"""GROCERY I"""
0.9,39007,51,2644,23,"""GROCERY I"""
0.95,24827,51,2312,22,"""GROCERY I"""


In [14]:
window = params.profiling.windows[1]
print(window.name)
coverage_by_window[window.name].describe()

ultimos_24_meses


statistic,store_nbr,item_nbr,first_sale,last_sale,n_days_with_sale,span_days,density_pair_span,density_window,store_covers_full_window
str,f64,f64,str,str,f64,f64,f64,f64,f64
"""count""",172769.0,172769.0,"""172769""","""172769""",172769.0,172769.0,172769.0,172769.0,172769.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0
"""mean""",27.841112,1.2200e6,"""2015-11-25 09:04:35.361899""","""2017-07-23 03:36:30.330441""",413.57695,606.772164,0.664356,0.578384,0.943769
"""std""",15.954217,582102.004082,null,null,233.193577,207.679253,0.274849,0.315738,null
"""min""",1.0,96995.0,"""2015-08-16""","""2015-08-16""",1.0,1.0,0.002801,0.001368,0.0
"""25%""",14.0,796395.0,"""2015-08-16""","""2017-08-12""",198.0,575.0,0.462277,0.295486,null
"""50%""",28.0,1.239911e6,"""2015-08-17""","""2017-08-15""",451.0,727.0,0.733242,0.634747,null
"""75%""",42.0,1.695875e6,"""2015-10-23""","""2017-08-15""",631.0,731.0,0.905609,0.870041,null
"""max""",54.0,2.127114e6,"""2017-08-15""","""2017-08-15""",727.0,731.0,1.0,1.0,1.0


In [15]:
window = params.profiling.windows[1]
print(f"{window.name} -- require_full_window=False")
count_candidates(
    coverage_by_window[window.name].lazy(),
    min_density=params.profiling.density_cuts,
    require_full_window=False,
    items=items,
)

ultimos_24_meses -- require_full_window=False


min_density,n_pairs,n_stores,n_items,n_families,top_family
f64,u32,u32,u32,u32,str
0.5,104695,54,3578,32,"""GROCERY I"""
0.6,91375,54,3402,32,"""GROCERY I"""
0.7,76216,54,3215,31,"""GROCERY I"""
0.8,58311,54,2998,25,"""GROCERY I"""
0.9,36222,53,2596,23,"""GROCERY I"""
0.95,22004,52,2124,22,"""GROCERY I"""


In [16]:
window = params.profiling.windows[1]
print(f"{window.name} -- require_full_window=True")
count_candidates(
    coverage_by_window[window.name].lazy(),
    min_density=params.profiling.density_cuts,
    require_full_window=True,
    items=items,
)

ultimos_24_meses -- require_full_window=True


min_density,n_pairs,n_stores,n_items,n_families,top_family
f64,u32,u32,u32,u32,str
0.5,98492,51,3332,32,"""GROCERY I"""
0.6,85956,51,3143,32,"""GROCERY I"""
0.7,71750,51,2950,31,"""GROCERY I"""
0.8,54902,51,2757,24,"""GROCERY I"""
0.9,34110,50,2382,21,"""GROCERY I"""
0.95,20674,49,1920,18,"""GROCERY I"""


## Resumo

- **Cobertura temporal**: 2013-01-01 a 2017-08-15, 54 lojas, 4.036 itens distintos, 174.685 pares loja-item observados em pelo menos uma linha, 125.497.040 linhas de venda.
- **Vendas negativas**: 7.795 linhas (0,0062% do total) — proporção baixa e concentrada; ver `profile_negatives_by_item` para os itens com maior proporção (candidatos a devolução recorrente, Sprint 3).
- **Vendas fracionárias**: 8.191.095 linhas (6,53% do total) — é o sinal de item vendido por peso; ver `profile_fractional_by_item` para os itens quase inteiramente fracionários (candidatos a `unit_of_sale = kg`, Sprint 3).
- **Disponibilidade de `onpromotion`**: 100% nulo até 2014-03 (inclusive); a partir de 2014-04-01 o campo fica 100% preenchido (0% nulo) e permanece assim até o fim da série, sem mês de transição parcial. Isso corrige uma suposição anterior ("meados de 2015") que havia entrado no comentário do `params.yaml` antes deste perfil rodar contra o dado real — o número medido é abril de 2014.
- **Não existe preço, custo nem saldo de estoque no Favorita.** Todo número de volume acima é em UNIDADES vendidas, nunca em R$ — não há como calcular margem ou capital empregado neste dataset sem arbitrar um dos três (decisão fora do escopo desta sprint).
- **Candidatos por corte de densidade**: ver as tabelas de `count_candidates` acima, para as duas janelas e com/sem `require_full_window`. Em ambas as janelas, com o corte mais permissivo (0.5) sobram dezenas de milhares de pares cobrindo praticamente todas as 54 lojas e mais de 3.000 itens em ~30 famílias distintas — o corte relevante para restringir a 1 loja / 150–300 SKUs fica para a Sprint 4, com base nestes números.